####1. Read data from the sales_sample.csv file and analyse to identify problems

In [0]:
sale_sample_raw_df = (spark.read.format("csv")
                      .option("header", "true")
                      .option("inferSchema", "true")
                      .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv"))



1.1 Define schema

In [0]:
#Easy way to define  Schema
file_schema = """
id integer,
name string,
dop string,
phone long,
amount string,
discount string
"""


1.2 Read data

In [0]:
sale_sample_raw_df2 = (spark.read.format("csv")
                      .schema(file_schema)
                      .option("header", "true")
                      .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv")
                      )



1.3 Describe the data

In [0]:
sale_sample_raw_df2.describe().display() 
"""Describe helps us to get the statistics of the data(Exploratory Analysis (many other ways to do it)) 
Now Check the data what is wrong with the data and how to fix it.
"""

1.4 List down the problems you want to fix
1. Convert id from integer to string and rename it as transaction_id.
2. Rename the name column to customer_name.
3. Convert the dop to date format and rename the column to date_of_purchase.
4. Rename the phone column to customer_phone
5. Convert the amount to a long value and filter out nulls and outlier values. 
6. Rename the column to purchase_amount
7. Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

####2. Prepare and clean the Dataframe using appropriate transformations

2.1 Transform

In [0]:

sales_df = (sale_sample_raw_df2.selectExpr(
    "cast(id as string) as transaction_id",
    "name as customer_name",
    "nvl(try_cast(dop as date), to_date(dop, 'dd-MM-yyyy')) as date_of_purchase",
    "cast(phone as string) as customer_phone",
    "cast(amount as long) as purchase_amount",
    "nvl(try_cast(discount as double), 0) as applied_discount"
)).filter("purchase_amount is not null and purchase_amount < 200000")

"""
Sometimes cast will not work as expected, so we can use try_cast to handle the errors.
"""



2.2 Verify statistics

In [0]:
sales_df.describe("purchase_amount","applied_discount").display()